# El pingüino promedio no vino a la junta

**Nivel:** principiante

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jcval94/narrative/blob/codex/corporate-data-narrative-lab/corporate-data-narrative-lab/outputs/notebooks/05-el-pinguino-promedio-no-vino.ipynb)

## Pregunta central

¿Una sola masa típica describe bien los registros o debemos mostrar la distribución por especie antes de publicar el cartel?

## Recreación narrativa

> **Marta:** "El cartel dirá: ‘Un pingüino pesa...’ y ponemos una cifra."
>
> **Nadia:** "¿Cuál pingüino?"
>
> **Marta:** "El promedio."
>
> **Tomás:** "Perfecto. ¿Le ponemos especie?"
>
> **Marta:** "No cabe."
>
> **Nadia:** "Entonces el dato cabe y el pingüino no."

*La escena es una recreación; las conclusiones provienen del dataset citado.*

## Fuente real

**Palmer Archipelago (Antarctica) penguin data — penguins.csv**, Palmer Station LTER; paquete palmerpenguins de Horst, Hill y Gorman. [Página de origen](https://allisonhorst.github.io/palmerpenguins/) · [datos](https://raw.githubusercontent.com/allisonhorst/palmerpenguins/main/inst/extdata/penguins.csv) · licencia: CC0 1.0 Universal.  
Consultado: 2026-07-18 · 344 filas · columnas usadas: `species`, `island`, `bill_length_mm`, `bill_depth_mm`, `flipper_length_mm`, `body_mass_g`, `sex`, `year`.

In [1]:
# @title Preparar los datos { display-mode: "form" }
especie = "Todas" # @param ["Todas", "Adelie", "Chinstrap", "Gentoo"]
bins = 18 # @param {type:"slider", min:6, max:40, step:1}
suavizado = 0.65 # @param {type:"slider", min:0.4, max:1.2, step:0.05}
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from IPython.display import Markdown, display
DATA_URL = "https://raw.githubusercontent.com/allisonhorst/palmerpenguins/main/inst/extdata/penguins.csv"
df_completo = pd.read_csv(DATA_URL)
df = df_completo if especie == "Todas" else df_completo.query("species == @especie")
df = df.dropna(subset=["body_mass_g"]).reset_index(drop=True)
display(Markdown(f"**Vista:** {especie} · **n={len(df)}** masas válidas · **bins={bins}** · **suavizado KDE={suavizado}**"))
df[["species", "island", "body_mass_g", "sex", "year"]].head()

**Vista:** Todas · **n=342** masas válidas · **bins=18** · **suavizado KDE=0.65**

,species,island,body_mass_g,sex,year
0,Adelie,Torgersen,3750.0,male,2007
1,Adelie,Torgersen,3800.0,female,2007
2,Adelie,Torgersen,3250.0,female,2007
3,Adelie,Torgersen,3450.0,female,2007
4,Adelie,Torgersen,3650.0,male,2007


## 1. Resumen numérico

> **Tomás:** "Dame el número correcto."
>
> **Nadia:** "Hay varios correctos; contestan preguntas distintas."
>
> **Marta:** "Dame el que ocupe menos caracteres."


**Pregunta:** ¿Qué cuentan media, mediana, moda, rango, varianza, desviación y percentiles sobre la misma masa?

**Conexión:** Punto de partida: una cifra cómoda quiere representar toda la tabla.

In [2]:
masa = df["body_mass_g"].dropna()
modas = masa.mode().tolist()
resumen = pd.Series({
    "n": len(masa), "media": masa.mean(), "mediana": masa.median(), "moda(s)": modas,
    "rango": masa.max() - masa.min(), "varianza muestral": masa.var(ddof=1),
    "desviación estándar muestral": masa.std(ddof=1),
    "p10 · p25 · p75 · p90": masa.quantile([.10, .25, .75, .90]).round(1).to_dict()})
resumen

n                                                                             342
media                                                                 4201.754386
mediana                                                                    4050.0
moda(s)                                                                  [3800.0]
rango                                                                      3600.0
varianza muestral                                                   643131.077327
desviación estándar muestral                                           801.954536
p10 · p25 · p75 · p90           {0.1: 3300.0, 0.25: 3550.0, 0.75: 4750.0, 0.9:...
dtype: object

In [3]:
# @title Explorar Resumen numérico { display-mode: "form" }
grupo = especie
q10, q25, q75, q90 = masa.quantile([.10, .25, .75, .90])
media, mediana = resumen["media"], resumen["mediana"]
fig = go.Figure()
fig.add_trace(go.Violin(x=masa, y=[grupo] * len(masa), orientation="h", name="Distribución", box_visible=False, meanline_visible=False, points="all", jitter=.22, pointpos=-1.25, fillcolor="#DCE8E6", line_color="#2F5D62", marker={"color": "#2F5D62", "size": 4, "opacity": .28}, hovertemplate="masa=%{x:,.0f} g<extra></extra>"))
fig.add_trace(go.Scatter(x=[masa.min(), masa.max()], y=[grupo, grupo], mode="lines", name="Rango", line={"color": "#173F5F", "width": 2}, hovertemplate="extremo=%{x:,.0f} g<extra>rango</extra>"))
fig.add_trace(go.Scatter(x=[q10, q90], y=[grupo, grupo], mode="lines+markers", name="p10–p90", line={"color": "#C86B3C", "width": 6}, marker={"color": "#C86B3C", "size": 8}, hovertemplate="percentil=%{x:,.0f} g<extra>p10–p90</extra>"))
fig.add_trace(go.Scatter(x=[q25, q75], y=[grupo, grupo], mode="lines+markers", name="p25–p75", line={"color": "#D99B2B", "width": 12}, marker={"color": "#D99B2B", "size": 10}, hovertemplate="cuartil=%{x:,.0f} g<extra>p25–p75</extra>"))
fig.add_trace(go.Scatter(x=[mediana], y=[grupo], mode="markers", name="Mediana", marker={"color": "#173F5F", "size": 14, "symbol": "circle-open", "line": {"width": 3}}, hovertemplate="mediana=%{x:,.0f} g<extra></extra>"))
fig.add_trace(go.Scatter(x=[media], y=[grupo], mode="markers", name="Media", marker={"color": "#C84B74", "size": 14, "symbol": "diamond"}, hovertemplate="media=%{x:,.1f} g<extra></extra>"))
fig.add_trace(go.Scatter(x=modas, y=[grupo] * len(modas), mode="markers", name="Moda(s)", marker={"color": "#6E5AA8", "size": 13, "symbol": "star"}, hovertemplate="moda=%{x:,.0f} g<extra></extra>"))
todos = [True] * 7
fig.update_layout(template="plotly_white", height=470, title=f"Resumen de la masa corporal<br><sup>{grupo} · n={len(masa)} · varianza y desviación muestrales</sup>", xaxis_title="Masa corporal (g)", yaxis_title="Distribución seleccionada", legend={"orientation": "h", "y": -0.22}, margin={"t": 105, "b": 110}, font={"family": "Arial", "color": "#173F5F"}, hoverlabel={"bgcolor": "#FFFFFF"}, updatemenus=[{"x": 1, "y": 1.19, "xanchor": "right", "buttons": [{"label": "Todo", "method": "update", "args": [{"visible": todos}]}, {"label": "Centro", "method": "update", "args": [{"visible": [True, False, False, False, True, True, True]}]}, {"label": "Dispersión", "method": "update", "args": [{"visible": [True, True, True, True, False, False, False]}]}]}])
fig.show()


display(Markdown("**Lo que muestra:** La media y la mediana ubican el centro; la moda señala el valor más repetido. Rango, varianza y desviación cuantifican dispersión, mientras p10–p90 muestra dónde cae el 80% central sin fingir que todos pesan igual."))

**Lo que muestra:** La media y la mediana ubican el centro; la moda señala el valor más repetido. Rango, varianza y desviación cuantifican dispersión, mientras p10–p90 muestra dónde cae el 80% central sin fingir que todos pesan igual.

## 2. Distribuciones

> **Nadia:** "La gráfica tiene más de un montón."
>
> **Tomás:** "¿Son dos tipos de promedio?"
>
> **Marta:** "No. Son los pingüinos reclamando su especie."


**Pregunta:** ¿Qué revelan histograma, densidad, sesgo, picos y elección de bins que el resumen no puede mostrar?

**Conexión:** Resumen numérico describió centro y dispersión, pero todavía no mostró la forma ni explicó por qué una sola cifra mezcla especies.

In [4]:
masa = df["body_mass_g"].dropna()
grid = np.linspace(masa.min(), masa.max(), 240)
ancho = suavizado * 1.06 * masa.std() * len(masa) ** (-1 / 5)
densidad = np.exp(-.5 * ((grid[:, None] - masa.to_numpy()) / ancho) ** 2).mean(1) / (ancho * np.sqrt(2 * np.pi))
picos = grid[1:-1][(densidad[1:-1] > densidad[:-2]) & (densidad[1:-1] >= densidad[2:])]
forma = pd.Series({"bins": bins, "suavizado KDE": suavizado, "sesgo": masa.skew(), "picos KDE": len(picos),
                   "ubicación de picos (g)": np.round(picos).astype(int).tolist()})
forma

bins                                      18
suavizado KDE                           0.65
sesgo                               0.470329
picos KDE                                  3
ubicación de picos (g)    [3649, 4643, 5426]
dtype: object

In [5]:
# @title Explorar Distribuciones { display-mode: "form" }
paleta = {"Adelie": "#2F5D62", "Chinstrap": "#C86B3C", "Gentoo": "#C84B74"}
fig = go.Figure()
grupos = list(df.groupby("species", observed=True))
for nombre, datos in grupos:
    fig.add_trace(go.Histogram(x=datos["body_mass_g"], nbinsx=bins, histnorm="probability density", name=nombre, opacity=.48, marker={"color": paleta[nombre], "line": {"color": "#FFFFFF", "width": .7}}, hovertemplate=f"{nombre}<br>masa=%{{x:,.0f}} g<br>densidad=%{{y:.5f}}<extra></extra>"))
fig.add_trace(go.Scatter(x=grid, y=densidad, mode="lines", name="Densidad KDE", line={"color": "#D99B2B", "width": 4}, fill="tozeroy", fillcolor="rgba(217,155,43,.10)", hovertemplate="masa=%{x:,.0f} g<br>densidad=%{y:.5f}<extra>KDE</extra>"))
fig.add_trace(go.Scatter(x=picos, y=np.interp(picos, grid, densidad), mode="markers+text", text=[f"pico {i+1}" for i in range(len(picos))], textposition="top center", name="Picos KDE", marker={"color": "#6E5AA8", "size": 11, "symbol": "diamond"}, hovertemplate="pico=%{x:,.0f} g<extra></extra>"))
n_hist = len(grupos)
fig.add_vline(x=resumen["media"], line={"color": "#C84B74", "width": 2, "dash": "dash"}, annotation_text="media", annotation_position="top right")
fig.add_vline(x=resumen["mediana"], line={"color": "#173F5F", "width": 2, "dash": "dot"}, annotation_text="mediana", annotation_position="top left")
fig.update_layout(template="plotly_white", barmode="overlay", height=520, title=f"Histograma y densidad de la masa corporal<br><sup>{especie} · n={len(masa)} · {bins} bins · KDE={suavizado} · sesgo={masa.skew():.2f}</sup>", xaxis_title="Masa corporal (g)", yaxis_title="Densidad de probabilidad", legend={"orientation": "h", "y": -0.22}, margin={"t": 110, "b": 115}, font={"family": "Arial", "color": "#173F5F"}, hoverlabel={"bgcolor": "#FFFFFF"}, updatemenus=[{"x": 1, "y": 1.19, "xanchor": "right", "buttons": [{"label": "Histograma + densidad", "method": "update", "args": [{"visible": [True] * (n_hist + 2)}]}, {"label": "Sólo histograma", "method": "update", "args": [{"visible": [True] * n_hist + [False, False]}]}, {"label": "Sólo densidad", "method": "update", "args": [{"visible": [False] * n_hist + [True, True]}]}]}])
fig.show()


display(Markdown("**Lo que muestra:** Con «Todas» y el suavizado inicial aparecen varios picos; el color por especie explica la mezcla. Cambiar bins o KDE altera la forma aparente, no las masas: usa ambos controles y el filtro para comprobar si un pico es estable."))

**Lo que muestra:** Con «Todas» y el suavizado inicial aparecen varios picos; el color por especie explica la mezcla. Cambiar bins o KDE altera la forma aparente, no las masas: usa ambos controles y el filtro para comprobar si un pico es estable.

## Cómo se conecta todo

El resumen ubica centro y dispersión; la distribución revela la forma. Al mezclar especies aparecen zonas distintas que una sola cifra comprime. Los bins cambian las barras, no los registros: confirma los picos con densidad y segmentos.

## Decisión

El cartel mostrará la distribución y resúmenes por especie, junto con n y faltantes. No usaremos una masa global como descripción universal ni como instrucción de campo.

**Regla:** Resume primero y mira la forma después; si mezclaste grupos, segmenta antes de decidir.